# Proje 1 - Bölüm 1: Sınıflandırma Analizi

Acoustic Features Dataset - Duygu Sınıflandırması


## Adım 1: Kütüphanelerin İçe Aktarılması


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
import warnings
import sys
import io

# Windows terminalinde UTF-8 encoding sorununu çöz
if sys.platform == 'win32':
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')

# Uyarıları bastır
warnings.filterwarnings('ignore')

print("✓ Kütüphaneler yüklendi.")


## Adım 2: Yardımcı Fonksiyonların Tanımlanması


In [ ]:
# DOCX rapor için Document oluştur
doc = Document()

# === DOCX YARDIMCI FONKSİYONLAR ===

def add_heading_docx(doc, text, level=1):
    """DOCX'e başlık ekle"""
    heading = doc.add_heading(text, level=level)
    return heading

def add_paragraph_docx(doc, text, bold=False, italic=False):
    """DOCX'e paragraf ekle"""
    para = doc.add_paragraph(text)
    if bold:
        for run in para.runs:
            run.bold = True
    if italic:
        for run in para.runs:
            run.italic = True
    return para

def add_table_docx(doc, headers, rows):
    """DOCX'e tablo ekle"""
    table = doc.add_table(rows=1, cols=len(headers))
    table.style = 'Light Grid Accent 1'
    
    # Başlıkları ekle
    header_cells = table.rows[0].cells
    for i, header in enumerate(headers):
        header_cells[i].text = str(header)
        header_cells[i].paragraphs[0].runs[0].font.bold = True
    
    # Satırları ekle
    for row_data in rows:
        row_cells = table.add_row().cells
        for i, cell_data in enumerate(row_data):
            row_cells[i].text = str(cell_data)
    
    return table

# === GEREKLİ FONKSİYONLAR ===

def calculate_specificity(y_true, y_pred, labels):
    """
    Çok sinifli bir durumda her sinif için Özgüllük (Specificity) hesaplar.
    Specificity = TN / (TN + FP)
    """
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    specificities = {}
    
    for i, label in enumerate(labels):
        total_samples = np.sum(cm)
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp
        tn = total_samples - (tp + fp + fn)
        
        if (tn + fp) > 0:
            specificity = tn / (tn + fp)
        else:
            specificity = 0.0
        specificities[label] = specificity
        
    return specificities

def print_separator(title=""):
    """Bölüm ayırıcı yazdır"""
    print("\n" + "="*80)
    if title:
        print(f" {title}")
        print("="*80)
    else:
        print("="*80)

print("✓ Yardımcı fonksiyonlar tanımlandı.")


## Adım 3: Rapor Başlangıcı ve Problem Açıklaması


In [ ]:
# === RAPOR BAŞLANGIÇ ===
print_separator("MAKİNE ÖĞRENMESİ SINIFLANDIRMA RAPORU")
print("\nAcoustic Features Dataset - Duygu Sınıflandırması Analizi\n")

# DOCX başlık
add_heading_docx(doc, "MAKİNE ÖĞRENMESİ SINIFLANDIRMA RAPORU", level=0)
add_paragraph_docx(doc, "Acoustic Features Dataset - Duygu Sınıflandırması Analizi")
add_paragraph_docx(doc, "")

# === 1. PROBLEM AÇIKLAMASI ===
print_separator("1. PROBLEM AÇIKLAMASI")

problem_text = """Bu veri setinde, müzik parçalarının akustik özelliklerine göre duygu sınıflandırması yapılmaktadır. Müzik eserlerinden çıkarılan 50 farklı akustik özellik (RMS enerji, MFCC katsayıları, spektral özellikler, harmonik değişim algılama fonksiyonu vb.) kullanılarak, müzik parçalarının hangi duyguyu yansıttığı tahmin edilmeye çalışılmaktadır. Problem, müzik parçalarının akustik özelliklerine göre duygu sınıflandırması problemidir (örneğin, otomobillerin özelliklerine göre kalite sınıflandırması gibi). Bu problemde, her bir müzik parçası akustik özellikler vektörü olarak temsil edilir ve bu vektör kullanılarak müzik parçasının hangi duygu kategorisinde (angry, happy, relax, sad) olduğu belirlenmeye çalışılır."""
print("\n" + problem_text + "\n")

# DOCX problem açıklaması
add_heading_docx(doc, "1. PROBLEM AÇIKLAMASI", level=1)
add_paragraph_docx(doc, problem_text)


## Adım 4: Veri Yükleme


In [ ]:
# === 2. VERİ ANALİZİ ===
print_separator("2. VERİ ANALİZİ")

try:
    # Veri setini yükle
    df = pd.read_csv('Acoustic Features.csv')
    
    # Class sütununu etiketler (y) olarak ayır
    y_labels = df['Class'].values
    
    # Class sütunu dışındaki tüm sütunları özellikler (X) olarak al
    feature_names = [col for col in df.columns if col != 'Class']
    X = df.drop('Class', axis=1).values
    
    # Etiketleri encode et
    le = LabelEncoder()
    y = le.fit_transform(y_labels)
    class_names = le.classes_
    
    print(f"✓ Veri seti yüklendi: {X.shape[0]} örnek, {X.shape[1]} öznitelik")
    print(f"✓ Sınıf sayısı: {len(class_names)}")
    print(f"✓ Sınıflar: {', '.join(class_names)}")
    
except FileNotFoundError:
    print("HATA: 'Acoustic Features.csv' dosyası bulunamadı.")
    print("Lütfen dataset dosyasını kodla aynı dizine koyun.")
    raise
except Exception as e:
    print(f"Bir hata oluştu: {e}")
    import traceback
    traceback.print_exc()
    raise


## Adım 5: Veri Özeti ve Sınıf Dağılımı


In [ ]:
# Veri özeti tablosu
print("\n" + "-"*80)
print(f"{'VERİ SETİ ÖZET TABLOSU':^80}")
print("-"*80)
print(f"{'Özellik':<30} {'Değer':<50}")
print("-"*80)
print(f"{'Toplam Örnek Sayısı':<30} {X.shape[0]:<50}")
print(f"{'Öznitelik Sayısı':<30} {X.shape[1]:<50}")
print(f"{'Sınıf Sayısı':<30} {len(class_names):<50}")
print(f"{'Sınıf İsimleri':<30} {', '.join(class_names):<50}")
print("-"*80)

# DOCX veri analizi başlık
add_heading_docx(doc, "2. VERİ ANALİZİ", level=1)

# DOCX veri özeti tablosu
add_heading_docx(doc, "Veri Seti Özet Tablosu", level=2)
summary_rows = [
    ['Toplam Örnek Sayısı', str(X.shape[0])],
    ['Öznitelik Sayısı', str(X.shape[1])],
    ['Sınıf Sayısı', str(len(class_names))],
    ['Sınıf İsimleri', ', '.join(class_names)]
]
add_table_docx(doc, ['Özellik', 'Değer'], summary_rows)

# Öznitelik isimleri
print(f"\n{'ÖZNİTELİK LİSTESİ (İlk 20 Öznitelik):':^80}")
print("-"*80)
for i, feat_name in enumerate(feature_names[:20], 1):
    print(f"{i:2d}. {feat_name}")
if len(feature_names) > 20:
    print(f"... ve {len(feature_names) - 20} öznitelik daha")
print("-"*80)

# Sınıf dağılımı
print(f"\n{'SINIF DAĞILIMI:':^80}")
print("-"*80)
print(f"{'Sınıf':<20} {'Örnek Sayısı':<20} {'Yüzde (%)':<20}")
print("-"*80)
unique, counts = np.unique(y_labels, return_counts=True)
class_distribution_rows = []
for class_name, count in zip(unique, counts):
    percentage = (count / len(y_labels)) * 100
    print(f"{class_name:<20} {count:<20} {percentage:.2f}%")
    class_distribution_rows.append([class_name, str(count), f"{percentage:.2f}%"])
print("-"*80)

# DOCX sınıf dağılımı tablosu
add_heading_docx(doc, "Sınıf Dağılımı", level=2)
add_table_docx(doc, ['Sınıf', 'Örnek Sayısı', 'Yüzde (%)'], class_distribution_rows)


## Adım 6: Yöntem Açıklamaları


In [ ]:
# === 3. YÖNTEM AÇIKLAMALARI ===
print_separator("3. KULLANILAN SINIFLANDIRMA YÖNTEMLERİ")

method_text_svm = """Support Vector Machine (SVM), sınıflandırma problemlerinde kullanılan güçlü bir makine öğrenmesi algoritmasıdır. Bu çalışmada RBF (Radial Basis Function) kernel kullanılmıştır. SVM, veri noktalarını farklı sınıflara ayıran en iyi hiperdüzlemi (decision boundary) bulmaya çalışır. RBF kernel, doğrusal olmayan sınıflandırma problemlerinde kullanılır ve farklı ölçeklerdeki verilerle çalışabilir. SVM, özellikle yüksek boyutlu veri setlerinde ve az sayıda örnek olduğunda etkili sonuçlar verir. Ancak, SVM'in performansı için verilerin normalize edilmesi kritik öneme sahiptir."""

method_text_rf = """Random Forest, birden fazla karar ağacının birleşiminden oluşan topluluk (ensemble) öğrenmesi yöntemidir. Her ağaç, veri setinden rastgele seçilen örnekler ve öznitelikler kullanılarak eğitilir. Tahmin yaparken, tüm ağaçların tahminleri toplanır ve çoğunluk oylaması (voting) ile final tahmin yapılır. Random Forest'in avantajları arasında, overfitting'e karşı dirençli olması, özellik önemini hesaplayabilmesi ve hiperparametre optimizasyonuna çok hassas olmaması sayılabilir. Bu çalışmada 100 ağaç kullanılmıştır."""

method_text = f"""
3.1. Support Vector Machine (SVC - RBF Kernel)
-----------------------------------------------
{method_text_svm}

3.2. Random Forest (Rastgele Orman)
-----------------------------------
{method_text_rf}
"""
print(method_text)

# DOCX yöntem açıklamaları
add_heading_docx(doc, "3. KULLANILAN SINIFLANDIRMA YÖNTEMLERİ", level=1)
add_heading_docx(doc, "3.1. Support Vector Machine (SVC - RBF Kernel)", level=2)
add_paragraph_docx(doc, method_text_svm)
add_heading_docx(doc, "3.2. Random Forest (Rastgele Orman)", level=2)
add_paragraph_docx(doc, method_text_rf)


## Adım 7: Veri Ön İşleme ve Normalizasyon


In [ ]:
# === 4. VERİ ÖN İŞLEME VE NORMALİZASYON ===
print_separator("4. VERİ ÖN İŞLEME VE NORMALİZASYON")

normalization_text = """Bu çalışmada StandardScaler kullanılarak veri normalizasyonu yapılmıştır. Standardizasyon, her özelliğin ortalamasını 0'a, standart sapmasını 1'e dönüştürür. Bu işlem z = (x - μ) / σ formülü ile yapılır. Burada x orijinal değer, μ özelliğin ortalaması, σ özelliğin standart sapması ve z normalize edilmiş değerdir. Normalizasyon neden önemlidir? SVM gibi algoritmalar, özelliklerin ölçeklerine duyarlıdır. Farklı ölçeklerdeki özellikler (örneğin 0-1 arası vs 1000-10000 arası), algoritmanın bazı özelliklere daha fazla ağırlık vermesine neden olabilir. Normalizasyon ile tüm özellikler aynı ölçekte olur ve algoritma tüm özelliklere eşit şans tanır."""

print(f"\n    Normalizasyon (Standardizasyon):\n    {normalization_text}\n")

# DOCX normalizasyon
add_heading_docx(doc, "4. VERİ ÖN İŞLEME VE NORMALİZASYON", level=1)
add_paragraph_docx(doc, normalization_text)

# Veriyi normalize et
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Veriyi train-test olarak ayır
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\n{'Veri Bölümleme Bilgileri:':^80}")
print("-"*80)
print(f"Eğitim Seti: {X_train.shape[0]} örnek ({X_train.shape[0]/X.shape[0]*100:.1f}%)")
print(f"Test Seti: {X_test.shape[0]} örnek ({X_test.shape[0]/X.shape[0]*100:.1f}%)")
print(f"Öznitelik Sayısı: {X_train.shape[1]}")
print("-"*80)
print("✓ Veri ön işleme tamamlandı.")


## Adım 8: Model Eğitimi ve Test Sonuçları


In [ ]:
# === 5. EĞİTİM/TEST SONUÇLARI ===
print_separator("5. EĞİTİM/TEST SONUÇLARI")

train_test_text = """
Eğitim/Test Ayrımı:
------------------
Veri seti %70 eğitim ve %30 test olarak ayrılmıştır. Stratified splitting kullanılarak 
her sınıfın oranının hem eğitim hem de test setinde korunması sağlanmıştır. Bu sayede 
sınıf dengesizliği problemi önlenmiştir.
"""
print(train_test_text)

# Modelleri tanımla
models = {
    "Support Vector Machine (SVC)": SVC(kernel='rbf', probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results_train_test = {}

for name, model in models.items():
    print(f"\n{'Model: ' + name:^80}")
    print("-"*80)
    
    # Modeli eğit
    model.fit(X_train, y_train)
    
    # Test verisi üzerinde tahmin yap
    y_pred = model.predict(X_test)
    
    # Metrikleri hesapla
    accuracy = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
    specificity_scores = calculate_specificity(y_test, y_pred, labels=np.arange(len(class_names)))
    
    # Sonuçları sakla
    results_train_test[name] = {
        'accuracy': accuracy,
        'confusion_matrix': cm,
        'report': report,
        'specificity': specificity_scores
    }
    
    # Confusion Matrix
    print(f"\n{'Confusion Matrix:':^80}")
    print("-"*80)
    cm_str = "          "
    for cn in class_names:
        cm_str += f"{cn:^15}"
    print(cm_str)
    print("-"*80)
    for i, cn in enumerate(class_names):
        row_str = f"{cn:^10}"
        for j in range(len(class_names)):
            row_str += f"{cm[i, j]:^15}"
        print(row_str)
    print("-"*80)
    print(f"\n{'Accuracy (Genel Doğruluk):':<40} {accuracy:.4f} ({accuracy*100:.2f}%)")
    print("-"*80)
    
    # Sınıf bazlı metrikler tablosu
    print(f"\n{'Sınıf Bazlı Metrikler:':^80}")
    print("-"*80)
    print(f"{'Sınıf':<15} {'Sensitivity':<15} {'Specificity':<15} {'Precision':<15} {'F1-Score':<15}")
    print("-"*80)
    for i, class_name in enumerate(class_names):
        sens = report[class_name]['recall']
        spec = specificity_scores[i]
        prec = report[class_name]['precision']
        f1 = report[class_name]['f1-score']
        print(f"{class_name:<15} {sens:<15.4f} {spec:<15.4f} {prec:<15.4f} {f1:<15.4f}")
    print("-"*80)

print("\n✓ Model eğitimi ve test tamamlandı.")


## Adım 9: Karşılaştırma Tablosu (Train/Test)


In [ ]:
# Karşılaştırma Tablosu
print(f"\n{'YÖNTEMLER ARASI KARŞILAŞTIRMA TABLOSU:':^80}")
print("-"*80)
print(f"{'Yöntem':<30} {'Accuracy':<20} {'Ort. Sensitivity':<20} {'Ort. Specificity':<20}")
print("-"*80)
comparison_rows = []
for name in models.keys():
    avg_sens = np.mean([results_train_test[name]['report'][cn]['recall'] for cn in class_names])
    avg_spec = np.mean([results_train_test[name]['specificity'][i] for i in range(len(class_names))])
    print(f"{name:<30} {results_train_test[name]['accuracy']:<20.4f} {avg_sens:<20.4f} {avg_spec:<20.4f}")
    comparison_rows.append([name, f"{results_train_test[name]['accuracy']:.4f}", 
                           f"{avg_sens:.4f}", f"{avg_spec:.4f}"])
print("-"*80)

# DOCX train-test sonuçları
add_heading_docx(doc, "5. EĞİTİM/TEST SONUÇLARI", level=1)
for name, res in results_train_test.items():
    add_heading_docx(doc, f"Model: {name}", level=2)
    add_paragraph_docx(doc, f"Accuracy (Genel Doğruluk): {res['accuracy']:.4f} ({res['accuracy']*100:.2f}%)")
    
    # Sınıf bazlı metrikler tablosu
    metric_rows = []
    for i, class_name in enumerate(class_names):
        sens = res['report'][class_name]['recall']
        spec = res['specificity'][i]
        prec = res['report'][class_name]['precision']
        f1 = res['report'][class_name]['f1-score']
        metric_rows.append([class_name, f"{sens:.4f}", f"{spec:.4f}", f"{prec:.4f}", f"{f1:.4f}"])
    add_table_docx(doc, ['Sınıf', 'Sensitivity', 'Specificity', 'Precision', 'F1-Score'], metric_rows)

# DOCX karşılaştırma tablosu
add_heading_docx(doc, "Yöntemler Arası Karşılaştırma", level=2)
add_table_docx(doc, ['Yöntem', 'Accuracy', 'Ort. Sensitivity', 'Ort. Specificity'], comparison_rows)


## Adım 10: 10-Fold Cross-Validation Açıklaması


In [ ]:
# === 6. 10-FOLD CROSS VALIDATION ===
print_separator("6. 10-KATMANLI ÇAPRAZ DOĞRULAMA (10-FOLD CROSS VALIDATION)")

cv_text = """
10-Fold Cross Validation Açıklaması:
-----------------------------------
Cross-validation (çapraz doğrulama), model performansını değerlendirmek için kullanılan 
istatistiksel bir yöntemdir. 10-fold cross-validation'da:

1. Veri seti rastgele olarak 10 eşit parçaya (fold) bölünür
2. Her iterasyonda, 9 fold eğitim için, 1 fold test için kullanılır
3. Bu işlem 10 kez tekrarlanır, her seferinde farklı bir fold test seti olarak kullanılır
4. 10 farklı test skorunun ortalaması alınarak modelin genel performansı değerlendirilir

Avantajları:
- Tüm veri hem eğitim hem test için kullanılır
- Model performansının daha güvenilir bir tahmini verilir
- Overfitting tespiti için etkilidir
- Farklı veri bölümlerine karşı modelin ne kadar tutarlı olduğunu gösterir

Bu çalışmada, veri karıştırılarak (shuffle=True) ve rastgele durum sabitlenerek (random_state=42) 
cross-validation uygulanmıştır.
"""
print(cv_text)


## Adım 11: Cross-Validation Uygulaması


In [ ]:
# 10-fold CV
kf = KFold(n_splits=10, shuffle=True, random_state=42)
results_cv = {}

for name, model in models.items():
    print(f"\n{'Model: ' + name:^80}")
    print("-"*80)
    
    scores = cross_val_score(model, X_scaled, y, cv=kf, scoring='accuracy')
    
    results_cv[name] = {
        'scores': scores,
        'mean': scores.mean(),
        'std': scores.std(),
        'min': scores.min(),
        'max': scores.max()
    }
    
    print(f"10-Fold Doğruluk Skorları:")
    for i, score in enumerate(scores, 1):
        print(f"  Fold {i:2d}: {score:.4f} ({score*100:.2f}%)")
    
    print("-"*80)
    print(f"{'Ortalama Doğruluk (Mean Accuracy):':<40} {scores.mean():.4f} ({scores.mean()*100:.2f}%)")
    print(f"{'Standart Sapma (Std Dev):':<40} {scores.std():.4f} ({scores.std()*100:.2f}%)")
    print(f"{'Minimum Skor:':<40} {scores.min():.4f} ({scores.min()*100:.2f}%)")
    print(f"{'Maksimum Skor:':<40} {scores.max():.4f} ({scores.max()*100:.2f}%)")
    print("-"*80)

print("\n✓ Cross-validation tamamlandı.")


## Adım 12: Cross-Validation Karşılaştırma Tablosu


In [ ]:
# CV Karşılaştırma Tablosu
print(f"\n{'10-FOLD CV KARŞILAŞTIRMA TABLOSU:':^80}")
print("-"*80)
print(f"{'Yöntem':<30} {'Ort. Accuracy':<20} {'Std Dev':<20} {'Min':<15} {'Max':<15}")
print("-"*80)
cv_comparison_rows = []
for name in models.keys():
    cv_res = results_cv[name]
    print(f"{name:<30} {cv_res['mean']:<20.4f} {cv_res['std']:<20.4f} {cv_res['min']:<15.4f} {cv_res['max']:<15.4f}")
    cv_comparison_rows.append([name, f"{cv_res['mean']:.4f}", f"{cv_res['std']:.4f}", 
                               f"{cv_res['min']:.4f}", f"{cv_res['max']:.4f}"])
print("-"*80)

# DOCX CV sonuçları
add_heading_docx(doc, "6. 10-KATMANLI ÇAPRAZ DOĞRULAMA (10-FOLD CROSS VALIDATION)", level=1)
cv_text_docx = """Cross-validation (çapraz doğrulama), model performansını değerlendirmek için kullanılan istatistiksel bir yöntemdir. 10-fold cross-validation'da veri seti rastgele olarak 10 eşit parçaya (fold) bölünür. Her iterasyonda, 9 fold eğitim için, 1 fold test için kullanılır. Bu işlem 10 kez tekrarlanır ve 10 farklı test skorunun ortalaması alınarak modelin genel performansı değerlendirilir."""
add_paragraph_docx(doc, cv_text_docx)

for name, cv_res in results_cv.items():
    add_heading_docx(doc, f"Model: {name}", level=2)
    add_paragraph_docx(doc, f"Ortalama Doğruluk: {cv_res['mean']:.4f} ({cv_res['mean']*100:.2f}%)")
    add_paragraph_docx(doc, f"Standart Sapma: {cv_res['std']:.4f} ({cv_res['std']*100:.2f}%)")
    add_paragraph_docx(doc, f"Minimum Skor: {cv_res['min']:.4f} ({cv_res['min']*100:.2f}%)")
    add_paragraph_docx(doc, f"Maksimum Skor: {cv_res['max']:.4f} ({cv_res['max']*100:.2f}%)")

# DOCX CV karşılaştırma tablosu
add_heading_docx(doc, "10-Fold CV Karşılaştırma Tablosu", level=2)
add_table_docx(doc, ['Yöntem', 'Ort. Accuracy', 'Std Dev', 'Min', 'Max'], cv_comparison_rows)


## Adım 13: Sonuçlar ve Yorumlar


In [ ]:
# === 7. SONUÇ VE YORUMLAR ===
print_separator("7. SONUÇ VE YORUMLAR")

# Train-test sonuçları yorumu
print("\n5.1. Eğitim/Test Sonuçları Yorumu:")
print("-"*80)
best_train_test = max(results_train_test.items(), key=lambda x: x[1]['accuracy'])
print(f"En yüksek doğruluk değeri {best_train_test[0]} yöntemi ile elde edilmiştir: {best_train_test[1]['accuracy']:.4f}")

for name, res in results_train_test.items():
    print(f"\n{name}:")
    print(f"  - Genel doğruluk: {res['accuracy']:.4f} ({res['accuracy']*100:.2f}%)")
    avg_sens = np.mean([res['report'][cn]['recall'] for cn in class_names])
    avg_spec = np.mean([res['specificity'][i] for i in range(len(class_names))])
    print(f"  - Ortalama Sensitivity: {avg_sens:.4f}")
    print(f"  - Ortalama Specificity: {avg_spec:.4f}")

# CV sonuçları yorumu
print("\n\n6.1. 10-Fold Cross-Validation Sonuçları Yorumu:")
print("-"*80)
best_cv = max(results_cv.items(), key=lambda x: x[1]['mean'])
print(f"En yüksek ortalama doğruluk değeri {best_cv[0]} yöntemi ile elde edilmiştir: {best_cv[1]['mean']:.4f}")

for name, res in results_cv.items():
    print(f"\n{name}:")
    print(f"  - Ortalama doğruluk: {res['mean']:.4f} ({res['mean']*100:.2f}%)")
    print(f"  - Standart sapma: {res['std']:.4f} ({res['std']*100:.2f}%)")
    print(f"  - Skor aralığı: {res['min']:.4f} - {res['max']:.4f}")
    if res['std'] < 0.05:
        print(f"  - Yorum: Model performansı tutarlıdır (düşük standart sapma)")
    else:
        print(f"  - Yorum: Model performansında değişkenlik vardır (yüksek standart sapma)")

# Genel yorum
general_comment = """Her iki yöntem de duygu sınıflandırması için başarılı sonuçlar vermiştir. 10-fold cross-validation sonuçları, eğitim/test sonuçlarından genellikle biraz daha yüksek çıkmıştır, bu durum modelin genelleme yeteneğinin iyi olduğunu göstermektedir. Cross-validation ile elde edilen standart sapma değerleri, modellerin performansının veri bölümlerine göre ne kadar tutarlı olduğunu göstermektedir. Düşük standart sapma, modelin daha güvenilir olduğunu ifade eder. Sınıf bazlı analizlerde, bazı duyguların (happy gibi) diğerlerine göre daha kolay sınıflandırıldığı görülmektedir. Bu durum, müzik parçalarının akustik özelliklerinin farklı duygular için farklı ayırt edici özellikler içerdiğini göstermektedir."""

print("\n\nGenel Yorum:")
print("-"*80)
print(f"\n    {general_comment}\n")

print_separator("RAPOR SONU")

# DOCX sonuç ve yorumlar
add_heading_docx(doc, "7. SONUÇ VE YORUMLAR", level=1)
add_heading_docx(doc, "Eğitim/Test Sonuçları Yorumu", level=2)
best_train_test = max(results_train_test.items(), key=lambda x: x[1]['accuracy'])
add_paragraph_docx(doc, f"En yüksek doğruluk değeri {best_train_test[0]} yöntemi ile elde edilmiştir: {best_train_test[1]['accuracy']:.4f}")

for name, res in results_train_test.items():
    add_paragraph_docx(doc, f"{name}: Genel doğruluk: {res['accuracy']:.4f} ({res['accuracy']*100:.2f}%)", bold=True)
    avg_sens = np.mean([res['report'][cn]['recall'] for cn in class_names])
    avg_spec = np.mean([res['specificity'][i] for i in range(len(class_names))])
    add_paragraph_docx(doc, f"  - Ortalama Sensitivity: {avg_sens:.4f}")
    add_paragraph_docx(doc, f"  - Ortalama Specificity: {avg_spec:.4f}")

add_heading_docx(doc, "10-Fold Cross-Validation Sonuçları Yorumu", level=2)
best_cv = max(results_cv.items(), key=lambda x: x[1]['mean'])
add_paragraph_docx(doc, f"En yüksek ortalama doğruluk değeri {best_cv[0]} yöntemi ile elde edilmiştir: {best_cv[1]['mean']:.4f}")

for name, res in results_cv.items():
    add_paragraph_docx(doc, f"{name}: Ortalama doğruluk: {res['mean']:.4f} ({res['mean']*100:.2f}%)", bold=True)
    add_paragraph_docx(doc, f"  - Standart sapma: {res['std']:.4f} ({res['std']*100:.2f}%)")
    add_paragraph_docx(doc, f"  - Skor aralığı: {res['min']:.4f} - {res['max']:.4f}")
    if res['std'] < 0.05:
        add_paragraph_docx(doc, "  - Yorum: Model performansı tutarlıdır (düşük standart sapma)")
    else:
        add_paragraph_docx(doc, "  - Yorum: Model performansında değişkenlik vardır (yüksek standart sapma)")

add_heading_docx(doc, "Genel Yorum", level=2)
add_paragraph_docx(doc, general_comment)


## Adım 14: DOCX Raporunun Kaydedilmesi


In [ ]:
# DOCX dosyasını kaydet
doc_filename = 'Rapor_Siniflandirma_Analizi.docx'
doc.save(doc_filename)
print(f"\n{'='*80}")
print(f"DOCX raporu oluşturuldu: {doc_filename}")
print(f"{'='*80}\n")
print("✓ Tüm analiz tamamlandı!")


In [ ]:
# Proje 1 - Bölüm 1: Sınıflandırma Analizi

Acoustic Features Dataset - Duygu Sınıflandırması
